In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
import os
os.getcwd()

'C:\\Users\\admin\\Desktop\\phishing-email-detection-main\\phishing-email-detection-main\\notebooks'

In [4]:
df = pd.read_csv("../data/enron_spam_data.csv")
df.head()

,Unnamed: 0,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14


In [5]:
import os
os.listdir("..")

['.ipynb_checkpoints',
 'anaconda_projects',
 'data',
 'notebooks',
 'README.md',
 'Untitled.ipynb']

In [6]:
os.listdir("../data")

['enron_spam_data.csv']

In [7]:
df = pd.read_csv("../data/enron_spam_data.csv")
df.head()

,Unnamed: 0,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14


In [8]:
df = df.drop(columns=["Unnamed: 0"])
df.head()

,Subject,Message,Spam/Ham,Date
0,christmas tree farm pictures,NaN,ham,1999-12-10
1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14


In [9]:
df = df.rename(columns={
    "Subject": "subject",
    "Message": "body",
    "Spam/Ham": "label_text",
    "Date": "date"
})

df.head()

,subject,body,label_text,date
0,christmas tree farm pictures,NaN,ham,1999-12-10
1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14


In [10]:
df.columns

Index(['subject', 'body', 'label_text', 'date'], dtype='object')

In [11]:
import re

def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("\r", " ").replace("\n", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

df["subject"] = df["subject"].apply(clean_text)
df["body"] = df["body"].apply(clean_text)

# Combine into one text column
df["text"] = (df["subject"] + " " + df["body"]).str.strip()

df[["subject", "body", "text"]].head()

,subject,body,text
0,christmas tree farm pictures,,christmas tree farm pictures
1,"vastar resources , inc .","gary , production from the high island larger ...","vastar resources , inc . gary , production fro..."
2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,calpine daily gas nomination - calpine daily g...
3,re : issue,fyi - see note below - already done . stella -...,re : issue fyi - see note below - already done...
4,meter 7268 nov allocation,fyi . - - - - - - - - - - - - - - - - - - - - ...,meter 7268 nov allocation fyi . - - - - - - - ...


In [12]:
df["label_text"].value_counts()

label_text
spam    17171
ham     16545
Name: count, dtype: int64

In [13]:
df["label"] = df["label_text"].map({"ham": 0, "spam": 1})
df[["label_text", "label"]].head()

,label_text,label
0,ham,0
1,ham,0
2,ham,0
3,ham,0
4,ham,0


In [14]:
df["label"].value_counts()

label
1    17171
0    16545
Name: count, dtype: int64

In [15]:
from sklearn.model_selection import train_test_split

X = df["text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training size:", len(X_train))
print("Testing size:", len(X_test))

Training size: 26972
Testing size: 6744


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Training vector shape:", X_train_vec.shape)
print("Testing vector shape:", X_test_vec.shape)

Training vector shape: (26972, 5000)
Testing vector shape: (6744, 5000)


In [17]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_vec, y_train)

print("Model training complete.")

Model training complete.


In [18]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.9989620403321471

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3309
           1       1.00      1.00      1.00      3435

    accuracy                           1.00      6744
   macro avg       1.00      1.00      1.00      6744
weighted avg       1.00      1.00      1.00      6744

